# V3 ablation 1 — no Structural Features, no Syntax-Aware Swap

Train only on HC3 train/validation and test only GPT-3.5 Mixed clean (0% noise). This is the 2×2 configuration `struct=0`, `aug=none`.

In [ ]:
!pip install -q --upgrade torch-geometric transformers spacy scikit-learn tqdm
!python -m spacy download en_core_web_sm
import torch, torch_geometric, transformers, spacy
print({'torch': torch.__version__, 'torch_geometric': torch_geometric.__version__, 'transformers': transformers.__version__, 'spacy': spacy.__version__})

In [ ]:
from pathlib import Path
import json, shutil, subprocess, sys

WORK = Path('/kaggle/working/v3_abla1_no_struct_no_swap_hc3_to_gpt35')
REPO = WORK / 'PRDetect-Research'
BRANCH = 'feat/rgcn-rdrop'
SEED = 2026
BATCH_SIZE = 64
MICRO_BATCH_SIZE = 8
EPOCHS = 10
VARIANT = 'baseline'
STRUCT_MODE = 'low'
STRUCT = '0'
AUG = 'none'
TEST_SPLIT = 'gpt3.5_mixed_test_split'
MODEL_DIR = WORK / 'model'
EVAL_DIR = WORK / 'evaluation'
if REPO.exists():
    shutil.rmtree(REPO)
WORK.mkdir(parents=True, exist_ok=True)
subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, 'https://github.com/tphan1420/PRDetect-Research.git', str(REPO)], check=True)
MODEL_DIR.mkdir(exist_ok=True); EVAL_DIR.mkdir(exist_ok=True)
assert (REPO / 'train_v3.py').is_file() and (REPO / 'evaluate_v3_quang.py').is_file()
assert (REPO / 'data/original_text/benchmarks/hc3/hc3_train.json').is_file()
assert (REPO / 'data/original_text/benchmarks/hc3/hc3_val.json').is_file()
assert (REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json').is_file()
assert not (REPO / 'graph_data' / f'{TEST_SPLIT}.pkl').exists(), 'Test graph must not exist before training.'
COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
print({'commit': COMMIT, 'abla': 1, 'struct': STRUCT, 'aug': AUG, 'seed': SEED, 'batch_size': BATCH_SIZE, 'test_only': TEST_SPLIT})

In [ ]:
subprocess.run([sys.executable, 'building_graph.py', '--file', 'hc3_train', 'hc3_val'], cwd=REPO, check=True)
TRAIN_GRAPH = REPO / 'graph_data/hc3_train.pkl'; VAL_GRAPH = REPO / 'graph_data/hc3_val.pkl'
assert TRAIN_GRAPH.is_file() and VAL_GRAPH.is_file()
assert not (REPO / 'graph_data' / f'{TEST_SPLIT}.pkl').exists(), 'Test graph was created before training.'

In [ ]:
subprocess.run([sys.executable, 'train_v3.py', '--variants', VARIANT, '--seeds', str(SEED), '--graph-dir', str(REPO / 'graph_data'), '--output-dir', str(MODEL_DIR), '--train-file', 'hc3_train', '--val-file', 'hc3_val', '--struct-mode', STRUCT_MODE, '--struct', STRUCT, '--aug', AUG, '--batch-size', str(BATCH_SIZE), '--micro-batch-size', str(MICRO_BATCH_SIZE), '--val-batch-size', '64', '--epochs', str(EPOCHS), '--patience', '10', '--device', 'cuda'], cwd=REPO, check=True)
CHECKPOINT = MODEL_DIR / f'v3_{VARIANT}_seed{SEED}.pt'; TRAIN_METADATA = CHECKPOINT.with_suffix('.train.json')
assert CHECKPOINT.is_file() and TRAIN_METADATA.is_file()

In [ ]:
subprocess.run([sys.executable, 'building_graph.py', '--file', TEST_SPLIT], cwd=REPO, check=True)
GRAPH = REPO / 'graph_data' / f'{TEST_SPLIT}.pkl'; assert GRAPH.is_file()
from types import SimpleNamespace
import importlib.util
spec = importlib.util.spec_from_file_location('v3_eval', REPO / 'evaluate_v3_quang.py'); v3_eval = importlib.util.module_from_spec(spec); spec.loader.exec_module(v3_eval)
args = SimpleNamespace(variant=VARIANT, train_domain='hc3', checkpoint_dir=str(MODEL_DIR), seeds=[SEED], batch_size=BATCH_SIZE, struct_mode=STRUCT_MODE, device='cuda', graph_budget_root=str(WORK))
CONFIG = EVAL_DIR / 'config.json'; v3_eval.write_json(CONFIG, v3_eval.make_config(args))
test_source = REPO / 'data/original_text/benchmarks/gpt35_mixed/gpt3.5_mixed_test_split.json'
expected_samples = sum(1 for line in test_source.open(encoding='utf-8') if line.strip()); assert expected_samples == 1000
REPORT = EVAL_DIR / 'gpt35_0pct_report.json'
subprocess.run([sys.executable, 'evaluate_v3_quang.py', '_test', '--config', str(CONFIG), '--seed', str(SEED), '--graph', str(GRAPH), '--expected-samples', str(expected_samples), '--signature', 'abla1_no_struct_no_swap_hc3_to_gpt35_0pct', '--report', str(REPORT)], cwd=REPO, check=True)
assert REPORT.is_file()

In [ ]:
import zipfile
report = json.loads(REPORT.read_text(encoding='utf-8'))
archive = WORK / f'v3_abla1_no_struct_no_swap_hc3_to_gpt35_0pct_seed{SEED}_batch{BATCH_SIZE}_results.zip'
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as out:
    for path in (CHECKPOINT, TRAIN_METADATA, CONFIG, REPORT): out.write(path, path.name)
print(json.dumps({key: report[key] for key in ('samples', 'accuracy', 'auc', 'f1', 'recall_ai', 'recall_human', 'threshold', 'validation_threshold')}, indent=2))
print('ZIP:', archive)